# Shortcut Learning in Skin Lesion Classifiers

Does a classifier trained on ISIC use the lesion, or the artifacts around it?

**Pipeline:** artifact correlation analysis -> baseline classifier -> paired
lesion/background ablation -> significance testing.

Runtime -> Change runtime type -> **T4 GPU** before running. Run cells in order.

## 0. Setup

In [ ]:
!pip install -q grad-cam

import os, random, numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
print('torch:', torch.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Everything persists here so a disconnect doesn't lose work
PROJECT = '/content/drive/MyDrive/shortcut-learning'
os.makedirs(PROJECT, exist_ok=True)
os.makedirs(f'{PROJECT}/checkpoints', exist_ok=True)
print(PROJECT)

## 1. Artifact correlation analysis

In [ ]:
!git clone -q --depth 1 https://github.com/alceubissoto/debiasing-skin.git /content/debiasing-skin

ann = pd.read_csv('/content/debiasing-skin/artefacts-annotation/isic_bias.csv',
                  sep=';', index_col=0)

ARTIFACTS = ['dark_corner','hair','gel_border','gel_bubble','ruler','ink','patches']

# strip .png so IDs join against the challenge .jpg files later
ann['image_id'] = ann['image'].str.replace('.png', '', regex=False)
ann['malignant'] = (ann['label_string'] == 'malignant').astype(int)

print(ann.shape)
ann.head()

In [ ]:
base = ann['malignant'].mean()

rows = []
for a in ARTIFACTS:
    present = ann[ann[a] == 1]
    absent  = ann[ann[a] == 0]
    rows.append({
        'artifact': a,
        'n_present': len(present),
        'prevalence': len(present) / len(ann),
        'malig_if_present': present['malignant'].mean(),
        'malig_if_absent': absent['malignant'].mean(),
        'lift': present['malignant'].mean() - base,
    })

corr_table = pd.DataFrame(rows).sort_values('lift', ascending=False)
print(f'Base malignancy rate: {base:.3f}  (n={len(ann)})\n')
display(corr_table.round(3))

corr_table.to_csv(f'{PROJECT}/artifact_correlations.csv', index=False)

In [ ]:
# Significance test 
# A lift is only worth reporting if it survives this.
from scipy.stats import chi2_contingency

for a in ARTIFACTS:
    table = pd.crosstab(ann[a], ann['malignant'])
    chi2, p, dof, _ = chi2_contingency(table)
    flag = '*' if p < 0.05 else ' '
    print(f'{a:12s} chi2={chi2:7.2f}  p={p:.2e} {flag}')

print('\n* = p < 0.05')

In [ ]:
# Figure 1
fig, ax = plt.subplots(figsize=(7, 4))
d = corr_table.sort_values('lift')
colors = ['#c0392b' if v > 0 else '#2874a6' for v in d['lift']]
ax.barh(d['artifact'], d['lift'], color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Malignancy rate lift vs. base rate')
ax.set_title(f'Artifact–label association, ISIC 2018 (base rate = {base:.1%})')
plt.tight_layout()
plt.savefig(f'{PROJECT}/fig1_artifact_lift.png', dpi=150)
plt.show()

## 2. Data

In [ ]:
DATA = '/content/data'
os.makedirs(DATA, exist_ok=True)

# unzip to local disk (much faster than reading from Drive during training)
!unzip -q -o {PROJECT}/raw/ISIC2018_Task1-2_Training_Input.zip -d {DATA}
!unzip -q -o {PROJECT}/raw/ISIC2018_Task1_Training_GroundTruth.zip -d {DATA}

IMG_DIR  = f'{DATA}/ISIC2018_Task1-2_Training_Input'
MASK_DIR = f'{DATA}/ISIC2018_Task1_Training_GroundTruth'

print('images:', len([f for f in os.listdir(IMG_DIR) if f.endswith('.jpg')]))
print('masks :', len([f for f in os.listdir(MASK_DIR) if f.endswith('.png')]))

In [ ]:
# Join annotations to files on disk. Verify nothing is missing before training.
def img_path(iid):  return f'{IMG_DIR}/{iid}.jpg'
def mask_path(iid): return f'{MASK_DIR}/{iid}_segmentation.png'

ann['has_img']  = ann['image_id'].apply(lambda i: os.path.exists(img_path(i)))
ann['has_mask'] = ann['image_id'].apply(lambda i: os.path.exists(mask_path(i)))

print('annotated rows :', len(ann))
print('with image     :', ann.has_img.sum())
print('with mask      :', ann.has_mask.sum())

df = ann[ann.has_img & ann.has_mask].reset_index(drop=True)
print('\nusable:', len(df))

# If 'usable' is much less than 2594, stop and check the mask filename pattern
print('\nsample mask filenames:', sorted(os.listdir(MASK_DIR))[:3])

In [ ]:
# Three-way split: train / val / test.
from sklearn.model_selection import train_test_split

df['n_artifacts'] = df[ARTIFACTS].sum(axis=1)
df['strat'] = df['malignant'].astype(str) + '_' + (df['n_artifacts'] > 0).astype(str)

trainval_df, test_df = train_test_split(
    df, test_size=0.20, stratify=df['strat'], random_state=SEED)

train_df, val_df = train_test_split(
    trainval_df, test_size=0.20, stratify=trainval_df['strat'], random_state=SEED)

for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f'{name:6s} n={len(d):5d}  malignant={d.malignant.mean():.3f}  '
          f'with-artifact={(d.n_artifacts > 0).mean():.3f}')

train_df.to_csv(f'{PROJECT}/train_split.csv', index=False)
val_df.to_csv(f'{PROJECT}/val_split.csv', index=False)
test_df.to_csv(f'{PROJECT}/test_split.csv', index=False)

## 3. Baseline classifier

Trains ResNet-18 with LR scheduling, checkpointing on best validation AUC, and
early stopping.

The goal is a *representative* model, not a strong one. Heavy tuning would make
the ablation results a function of the tuning rather than of the phenomenon.

In [ ]:
IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(180),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

# No augmentation at eval
# Ablation depends on deterministic predictions
eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

class LesionDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        img = Image.open(img_path(row.image_id)).convert('RGB')
        return self.transform(img), torch.tensor(row.malignant, dtype=torch.float32)

train_loader = DataLoader(LesionDataset(train_df, train_tf), batch_size=32,
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(LesionDataset(val_df, eval_tf), batch_size=32,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(LesionDataset(test_df, eval_tf), batch_size=32,
                          shuffle=False, num_workers=2, pin_memory=True)

print('batches — train:', len(train_loader), 'val:', len(val_loader),
      'test:', len(test_loader))

In [ ]:
import copy
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 1)
model = model.to(DEVICE)

pos_weight = torch.tensor([(1 - train_df.malignant.mean()) / train_df.malignant.mean()])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=1, min_lr=1e-7)

print('pos_weight:', round(pos_weight.item(), 2))

In [ ]:
def predict_all(model, loader):
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            p = torch.sigmoid(model(x.to(DEVICE))).squeeze(1).cpu()
            probs.append(p); labels.append(y)
    return torch.cat(probs).numpy(), torch.cat(labels).numpy()


def evaluate(model, loader, threshold=0.5):
    probs, labels = predict_all(model, loader)
    return {
        'auc': roc_auc_score(labels, probs),
        'acc': ((probs > threshold) == labels).mean(),
        'bal_acc': balanced_accuracy_score(labels, probs > threshold),
        'mean_prob': probs.mean(),          # exposes calibration drift
    }

MAX_EPOCHS = 40
PATIENCE = 8
MIN_DELTA = 1e-3

best_auc = -1.0
best_epoch = -1
best_state = None
epochs_without_improvement = 0
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    # train 
    model.train()
    running = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x).squeeze(1), y)
        loss.backward()
        optimizer.step()
        running += loss.item() * x.size(0)
    train_loss = running / len(train_loader.dataset)

    # validate (NEVER the test set) 
    v = evaluate(model, val_loader)
    current_lr = optimizer.param_groups[0]['lr']
    history.append({'epoch': epoch, 'train_loss': train_loss,
                    'lr': current_lr, **v})

    # LR schedule on val AUC 
    scheduler.step(v['auc'])

    # checkpoint on val AUC 
    improved = v['auc'] > best_auc + MIN_DELTA
    if improved:
        best_auc = v['auc']
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, f'{PROJECT}/checkpoints/resnet18_best.pth')
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(f'epoch {epoch:2d}  loss={train_loss:.4f}  lr={current_lr:.1e}  '
          f'val AUC={v["auc"]:.3f}  bal_acc={v["bal_acc"]:.3f}  '
          f'mean_p={v["mean_prob"]:.3f}' + ('  <- best' if improved else ''))

    # early stop 
    if epochs_without_improvement >= PATIENCE:
        print(f'\nNo val AUC improvement for {PATIENCE} epochs. Stopping.')
        break

# restore best weights, not last 
model.load_state_dict(best_state)
print(f'\nRestored epoch {best_epoch} (val AUC {best_auc:.3f})')

pd.DataFrame(history).to_csv(f'{PROJECT}/training_history.csv', index=False)

In [ ]:
val_probs, val_labels = predict_all(model, val_loader)

grid = np.linspace(0.05, 0.95, 91)
scores = [balanced_accuracy_score(val_labels, val_probs > t) for t in grid]
best_threshold = float(grid[int(np.argmax(scores))])
print(f'threshold chosen on val: {best_threshold:.2f} '
      f'(val bal_acc {max(scores):.3f})')

# the single test evaluation 
test_metrics = evaluate(model, test_loader, threshold=best_threshold)
print('\nTEST (reported once):')
for k, v_ in test_metrics.items():
    print(f'  {k:10s} {v_:.4f}')

pd.DataFrame([{**test_metrics, 'threshold': best_threshold,
               'best_epoch': best_epoch, 'val_auc': best_auc}]
             ).to_csv(f'{PROJECT}/test_metrics.csv', index=False)

In [ ]:
# This plot shows why the oscillation in the first run was a calibration artifact, not instability.
h = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(h.epoch, h.train_loss, marker='o', label='train loss')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss'); axes[0].legend()
axes[0].set_title('Training loss')

axes[1].plot(h.epoch, h.auc, marker='o', label='val AUC')
axes[1].plot(h.epoch, h.acc, marker='s', label='val acc @0.5')
axes[1].plot(h.epoch, h.bal_acc, marker='^', label='val balanced acc')
axes[1].axvline(best_epoch, ls='--', c='gray', lw=1)
axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].set_ylim(0.4, 1.0)
axes[1].set_title('Validation metrics (dashed = restored epoch)')

plt.tight_layout()
plt.savefig(f'{PROJECT}/training_curves.png', dpi=150)
plt.show()

## 4. The masking delta — the core measurement

`delta = p(original) − p(masked)`. Large delta means the model depended on what you covered.

Blur fill, not black — a black rectangle is out-of-distribution, so a confidence drop would just mean "weird input".

In [ ]:
def predict_prob(model, pil_img):
    model.eval()
    x = eval_tf(pil_img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        return torch.sigmoid(model(x)).item()

def load_mask(image_id, size):
    m = Image.open(mask_path(image_id)).convert('L').resize(size, Image.NEAREST)
    return np.array(m) > 127

def apply_mask(pil_img, keep_bool, fill='blur'):
    arr = np.array(pil_img).astype(np.float32)
    if fill == 'blur':
        rep = np.array(pil_img.filter(ImageFilter.GaussianBlur(25))).astype(np.float32)
    elif fill == 'mean':
        rep = np.zeros_like(arr) + arr.reshape(-1, 3).mean(axis=0)
    else:
        rep = np.zeros_like(arr)
    out = np.where(keep_bool[..., None], arr, rep)
    return Image.fromarray(out.astype(np.uint8))

def deltas_for(model, image_id, fill='blur'):
    img = Image.open(img_path(image_id)).convert('RGB')
    lesion = load_mask(image_id, img.size)

    p_orig = predict_prob(model, img)
    p_no_lesion = predict_prob(model, apply_mask(img, ~lesion, fill))  # lesion removed
    p_no_bg     = predict_prob(model, apply_mask(img,  lesion, fill))  # background removed

    return {
        'image_id': image_id,
        'p_orig': p_orig,
        'p_lesion_masked': p_no_lesion,
        'p_background_masked': p_no_bg,
        'delta_lesion': p_orig - p_no_lesion,
        'delta_background': p_orig - p_no_bg,
    }

In [ ]:
# Smoke test on 5 images
for iid in test_df.image_id.head(5):
    r = deltas_for(model, iid)
    print(f"{r['image_id']}  p={r['p_orig']:.3f}  "
          f"Δlesion={r['delta_lesion']:+.3f}  Δbackground={r['delta_background']:+.3f}")

In [ ]:
# Visual sanity check
iid = test_df.image_id.iloc[0]
img = Image.open(img_path(iid)).convert('RGB')
lesion = load_mask(iid, img.size)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (im, title) in zip(axes, [
        (img, 'original'),
        (apply_mask(img, ~lesion), 'lesion masked'),
        (apply_mask(img,  lesion), 'background masked')]):
    ax.imshow(im); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()


## 5. Full run + the occlusion check

Bissoto et al. found models classify above chance with the lesion fully removed. Test that on your model — if it holds, it's your headline result.

In [ ]:
from tqdm import tqdm

results = [deltas_for(model, iid) for iid in tqdm(test_df.image_id)]
res = pd.DataFrame(results).merge(
    test_df[['image_id', 'malignant'] + ARTIFACTS], on='image_id')
res.to_csv(f'{PROJECT}/ablation_results.csv', index=False)

print('Mean |delta| — lesion vs background:')
print(f"  lesion masked     : {res.delta_lesion.abs().mean():.4f}")
print(f"  background masked : {res.delta_background.abs().mean():.4f}")

# Occlusion check: can the model still separate classes with no lesion?
auc_no_lesion = roc_auc_score(res.malignant, res.p_lesion_masked)
print(f"\nAUC with lesion removed: {auc_no_lesion:.3f}  (0.5 = chance)")

In [ ]:
# Paired test: is background-masking different from lesion-masking?
from scipy.stats import wilcoxon

stat, p = wilcoxon(res.delta_lesion.abs(), res.delta_background.abs())
print(f'Wilcoxon signed-rank: stat={stat:.1f}  p={p:.2e}')

print('\nMean |delta_background| by artifact:')
for a in ARTIFACTS:
    sub = res[res[a] == 1]
    if len(sub) > 10:
        print(f'  {a:12s} n={len(sub):4d}  {sub.delta_background.abs().mean():.4f}')

## 6. Grad-CAM (illustration, not evidence)

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import BinaryClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

cam = GradCAM(model=model, target_layers=[model.layer4[-1]])

def show_gradcam(image_id):
    img = Image.open(img_path(image_id)).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    rgb = np.array(img).astype(np.float32) / 255.0
    x = eval_tf(img).unsqueeze(0).to(DEVICE)
    grayscale = cam(input_tensor=x, targets=[BinaryClassifierOutputTarget(1)])[0]
    overlay = show_cam_on_image(rgb, grayscale, use_rgb=True)

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    axes[0].imshow(img); axes[0].set_title(image_id); axes[0].axis('off')
    axes[1].imshow(overlay); axes[1].set_title('Grad-CAM'); axes[1].axis('off')
    plt.tight_layout(); plt.show()

# Pick images with an artifact so the figure is informative
for iid in test_df[test_df.ruler == 1].image_id.head(3):
    show_gradcam(iid)